In [1]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append(r"Z:\HTOC\Data_Analytics\threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\HTOC\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    display(f"Loaded config from: {config_path}")
    display(f"Base URL: {api_base_url}")
    display(f"Access ID: {api_access_id}")
    display(f"Default Org: {api_default_org}")
except Exception as e:
    display(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    display("ThreatConnect initialized.")
except Exception as e:
    display(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    display("RequestObject successfully created.")
except Exception as e:
    display(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)




'Loaded config from: C:\\Users\\jaskew\\Documents\\HTOC\\scripts\\Data Movement\\ThrearConnect-api-pull\\utils\\config.json'

'Base URL: https://hvs.threatconnect.com/api'

'Access ID: 09783848890162390382'

'Default Org: HTOC Org'

'ThreatConnect initialized.'

'RequestObject successfully created.'

In [2]:
import pandas as pd
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
import contextlib
import io
import pytz
import urllib.parse

QUERY_LOOKBACK_DAYS = 365
OWNER_NAMES = [
    'HTOC Org'
]
INDICATOR_PAGE_SIZE = 10000
ATTRIBUTE_PAGE_SIZE = 10000
INDICATOR_WORKERS = 3
ATTRIBUTE_WORKERS = 3

start_date = (datetime.now(pytz.UTC) - timedelta(days=QUERY_LOOKBACK_DAYS)).date()
start = f"{start_date}T00:00:00Z"

owner_condition = ", ".join([f'"{o}"' for o in OWNER_NAMES])

indicator_tql = urllib.parse.quote(
    f'ownerName IN ({owner_condition}) AND '
    f'typeName IN ("Address") AND '
    f'dateAdded >= "2025-04-01" AND '
    f'NOT hasTag(summary IN ("SOAR Indicator PB"))'
)

attribute_tql = urllib.parse.quote(
    f'ownerName IN ({owner_condition})'
)

# ── Shared fetch helper ───────────────────────────────────────────────────────

def fetch(uri):
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_request_uri(uri)
    with contextlib.redirect_stdout(io.StringIO()), \
         contextlib.redirect_stderr(io.StringIO()):
        response = tc.api_request(ro)
    ct = response.headers.get('content-type', '')
    if not ct.startswith('application/json'):
        raise RuntimeError(f"Non-JSON response ({ct}): {response.content[:200]}")
    return response.json()

def fetch_all_pages(base_uri, page_size, workers):
    """Fetch first page, then remaining pages in parallel. Returns flat list of items.
    Falls back to sequential pagination when the API does not return a count field.
    """
    first = fetch(f"{base_uri}&resultStart=0&resultLimit={page_size}")
    total = first.get('count', 0)
    items = [i for i in first.get('data', []) if isinstance(i, dict)]

    if total > 0:
        # API returned a count — use it to build all offsets and fetch in parallel
        remaining = range(page_size, total, page_size)
        if remaining:
            with ThreadPoolExecutor(max_workers=workers) as executor:
                futures = {
                    executor.submit(fetch, f"{base_uri}&resultStart={o}&resultLimit={page_size}"): o
                    for o in remaining
                }
                for future in as_completed(futures):
                    try:
                        page = future.result()
                        items.extend(i for i in page.get('data', []) if isinstance(i, dict))
                    except Exception as e:
                        display(f"Page fetch failed (offset {futures[future]}): {e}")
    else:
        # No count returned — page sequentially until we get a partial page
        offset = page_size
        while len(items) % page_size == 0 and len(items) > 0:
            try:
                page = fetch(f"{base_uri}&resultStart={offset}&resultLimit={page_size}")
                batch = [i for i in page.get('data', []) if isinstance(i, dict)]
                if not batch:
                    break
                items.extend(batch)
                offset += page_size
                if len(batch) < page_size:
                    break
            except Exception as e:
                display(f"Page fetch failed (offset {offset}): {e}")
                break
        total = len(items)

    return total, items

# ── Phase 1: Indicators ───────────────────────────────────────────────────────

display("Fetching indicators...")
total_indicators, indicator_items = fetch_all_pages(
    f'/v3/indicators?tql={indicator_tql}&fields=tags,observations,associatedGroups',
    INDICATOR_PAGE_SIZE,
    INDICATOR_WORKERS,
)
display(f"Indicators fetched: {len(indicator_items)} of {total_indicators}")

observed_src = pd.json_normalize(indicator_items)
observed_src['indicator'] = observed_src['summary'].str.split(n=1).str[0].str.strip()

# ── Phase 2: Attributes (parallel per-indicator) ─────────────────────────────

def fetch_attributes(indicator):
    try:
        ro = RequestObject()
        ro.set_http_method('GET')
        ro.set_request_uri(
            f'/v3/indicators/{indicator}?fields=attributes'
            f'&resultStart=0&resultLimit=1000'
        )
        with contextlib.redirect_stdout(io.StringIO()), \
             contextlib.redirect_stderr(io.StringIO()):
            response = tc.api_request(ro)
        ct = response.headers.get('content-type', '')
        if not ct.startswith('application/json'):
            return indicator, [], True

        data = response.json().get('data', {})
        attributes = data.get('attributes', {}).get('data', [])
        if not attributes:
            return indicator, [], True

        enriched = []
        for attr in attributes:
            attr.update({
                'id': data.get('id'),
                'summary': data.get('summary'),
                'type': data.get('type'),
                'ownerName': data.get('ownerName'),
            })
            enriched.append(attr)
        return indicator, enriched, False

    except Exception as e:
        if hasattr(e, 'response') and getattr(e.response, 'status_code', None) == 400:
            return indicator, [], True
        if "Status Code: 400" in str(e):
            return indicator, [], True
        return indicator, [], True

unique_indicators = observed_src['summary'].unique()
display(f"Fetching attributes for {len(unique_indicators)} unique indicators...")

attributes_data = []
indicators_with_no_attributes = []

with ThreadPoolExecutor(max_workers=ATTRIBUTE_WORKERS) as executor:
    futures = {executor.submit(fetch_attributes, ind): ind for ind in unique_indicators}
    for future in as_completed(futures):
        indicator, attrs, no_attr = future.result()
        if no_attr:
            indicators_with_no_attributes.append(indicator)
        else:
            attributes_data.extend(attrs)

display(f"Attributes found: {len(attributes_data)} | No attributes: {len(indicators_with_no_attributes)}")

# ── Phase 3: Normalize & filter ───────────────────────────────────────────────

attributes_observed_src = pd.json_normalize(attributes_data)
attributes_observed_src = attributes_observed_src.drop_duplicates(subset='id').reset_index(drop=True)

filtered_with_attrs = observed_src[observed_src['summary'].isin(attributes_observed_src['summary'])]
no_attrs_df = observed_src[observed_src['summary'].isin(indicators_with_no_attributes)]

filtered_recent_tags = (
    pd.concat([filtered_with_attrs, no_attrs_df], ignore_index=True)
    .drop_duplicates(subset='summary')
    .reset_index(drop=True)
)

display(filtered_recent_tags)
display(attributes_observed_src)

'Fetching indicators...'

'Indicators fetched: 6668 of 6668'

'Fetching attributes for 6668 unique indicators...'

'Attributes found: 7994 | No attributes: 1530'

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,summary,...,privateFlag,active,activeLocked,ip,legacyLink,tags.data,associatedGroups.data,description,source,indicator
0,5629499583003470,2026-01-12T13:46:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-04-13T12:00:55Z,5.0,60.0,195.206.234.19,...,False,True,False,195.206.234.19,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 505199, 'name': 'Berserk Bear', 'lastU...","[{'id': 5629499569000349, 'dateAdded': '2026-0...",Summary\nRussian actors use obfuscated infrast...,NaN,195.206.234.19
1,5629499574089568,2025-10-21T11:33:56Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-04-13T11:19:02Z,5.0,76.0,23.106.169.120,...,False,True,False,23.106.169.120,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 1739147, 'name': 'OtterCookie', 'lastU...","[{'id': 6755399468000794, 'dateAdded': '2025-1...",Key Findings\nSilent Push Threat Analysts have...,NaN,23.106.169.120
2,5629499558103799,2025-07-09T16:54:02Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-04-13T11:19:01Z,5.0,62.0,104.28.211.105,...,False,True,False,104.28.211.105,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 1640817, 'name': 'SPACEHOP ORB', 'last...","[{'id': 6755399457034470, 'dateAdded': '2025-0...",Details\nThe attached IP addresses were nodes ...,NaN,104.28.211.105
3,5629499542036630,2025-05-28T19:30:47Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-04-13T11:19:00Z,3.0,35.0,193.163.125.52,...,False,True,False,193.163.125.52,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 2076981, 'name': 'SOAR Indicator PB', ...",NaN,RITM0584685 / TASK0881439,NaN,193.163.125.52
4,5629499542014600,2025-05-21T13:34:46Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-04-13T11:19:00Z,3.0,35.0,176.65.149.182,...,False,True,False,176.65.149.182,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...",NaN,RITM0582833,NaN,176.65.149.182
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6663,5629499536025730,2025-04-05T11:46:20Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-04-07T00:06:20Z,NaN,NaN,44.195.74.126,...,False,True,False,44.195.74.126,https://hvs.threatconnect.com/auth/indicators/...,NaN,NaN,NaN,NaN,44.195.74.126
6664,5629499536023506,2025-04-01T10:26:20Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-04-05T11:46:20Z,NaN,NaN,54.83.253.213,...,False,True,False,54.83.253.213,https://hvs.threatconnect.com/auth/indicators/...,NaN,NaN,NaN,NaN,54.83.253.213
6665,5629499536023505,2025-04-01T10:26:20Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-04-05T11:46:20Z,NaN,NaN,3.223.164.209,...,False,True,False,3.223.164.209,https://hvs.threatconnect.com/auth/indicators/...,NaN,NaN,NaN,NaN,3.223.164.209
6666,5629499536025509,2025-04-04T15:13:41Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-04-04T15:13:50Z,0.0,0.0,23.106.215.197,...,False,True,False,23.106.215.197,https://hvs.threatconnect.com/auth/indicators/...,NaN,"[{'id': 5629499536007279, 'dateAdded': '2025-0...",NaN,NaN,23.106.215.197


,id,dateAdded,type,value,lastModified,pinned,default,summary,ownerName,createdBy.id,createdBy.userName,createdBy.firstName,createdBy.lastName,createdBy.pseudonym,createdBy.owner
0,5629499583003470,2026-01-12T13:46:11Z,Address,Summary\nRussian actors use obfuscated infrast...,2026-01-12T13:47:15Z,False,True,195.206.234.19,HTOC Org,633,12488526555309068164,HTOC,Playbook User,APIUserOQys7,HTOC Org
1,5629499574089568,2025-10-21T11:33:56Z,Address,Key Findings\nSilent Push Threat Analysts have...,2025-10-21T11:33:56Z,False,True,23.106.169.120,HTOC Org,513,chase.nelson@hhs.gov,Chase,Nelson /HHS-HTOC/,cnelson-htoc-alyn,HTOC Org
2,5629499558103799,2025-07-09T16:54:02Z,Address,Details\nThe attached IP addresses were nodes ...,2025-07-09T16:54:02Z,False,True,104.28.211.105,HTOC Org,513,chase.nelson@hhs.gov,Chase,Nelson /HHS-HTOC/,cnelson-htoc-alyn,HTOC Org
3,5629499542036630,2025-05-28T19:30:47Z,Address,RITM0584685 / TASK0881439,2025-05-28T19:30:47Z,False,True,193.163.125.52,HTOC Org,6000,43204519559205009624,NIH,SOAR,APIUser7Lnh3,HTOC Org
4,5629499542014600,2025-05-21T13:34:46Z,Address,RITM0582833,2025-05-21T13:34:46Z,False,True,176.65.149.182,HTOC Org,6000,43204519559205009624,NIH,SOAR,APIUser7Lnh3,HTOC Org
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5133,5629499537000363,2025-04-21T13:15:31Z,Address,Indicator associated with Asylum Ambuscade/TA866,2025-04-21T13:15:31Z,False,True,22.113.16.27,HTOC Org,511,rodger.wille@hhs.gov,Rodger,Wille /HHS-HTOC/,RWille-HTOC-Alyn,HTOC Org
5134,5629499537000361,2025-04-21T13:15:31Z,Address,Indicator associated with Asylum Ambuscade/TA866,2025-04-21T13:15:31Z,False,True,194.76.225.67,HTOC Org,511,rodger.wille@hhs.gov,Rodger,Wille /HHS-HTOC/,RWille-HTOC-Alyn,HTOC Org
5135,5629499537000352,2025-04-21T13:15:31Z,Address,Indicator associated with Asylum Ambuscade/TA866,2025-04-21T13:15:31Z,False,True,23.227.193.25,HTOC Org,511,rodger.wille@hhs.gov,Rodger,Wille /HHS-HTOC/,RWille-HTOC-Alyn,HTOC Org
5136,5629499537000353,2025-04-21T13:15:31Z,Address,Indicator associated with Asylum Ambuscade/TA866,2025-04-21T13:15:31Z,False,True,37.1.212.198,HTOC Org,511,rodger.wille@hhs.gov,Rodger,Wille /HHS-HTOC/,RWille-HTOC-Alyn,HTOC Org


In [5]:
import os

# Filter for records with createdBy.firstName == 'NIH' and createdBy.lastName == 'SOAR'
nih_soar_records = attributes_observed_src[
    (attributes_observed_src['createdBy.firstName'] == 'NIH') &
    (attributes_observed_src['createdBy.lastName'] == 'SOAR')
]

# One row per indicator (same pattern as elsewhere in this notebook)
nih_soar_records = nih_soar_records.drop_duplicates(subset='summary').reset_index(drop=True)

out_dir = r"Z:\HTOC\JA\Documents\SOARIndicators"
os.makedirs(out_dir, exist_ok=True)
if 'associatedGroups.data' in nih_soar_records.columns:
    # Extract associatedGroups (e.g. group names as comma-separated values)
    nih_soar_records['associatedGroups'] = (
        nih_soar_records['associatedGroups.data'].apply(
            lambda ag: ', '.join(
                {g['name'] for g in ag if isinstance(g, dict) and 'name' in g}
            ) if isinstance(ag, list) else None
        )
    )
else:
    nih_soar_records['associatedGroups'] = None
out_path = os.path.join(out_dir, "nih_soar_indicators.csv")
nih_soar_records.to_csv(out_path, index=False)

display(f"Saved {len(nih_soar_records)} unique rows to {out_path}")
display(nih_soar_records)

'Saved 1912 unique rows to Z:\\HTOC\\JA\\Documents\\SOARIndicators\\nih_soar_indicators.csv'

,id,dateAdded,type,value,lastModified,pinned,default,summary,ownerName,createdBy.id,createdBy.userName,createdBy.firstName,createdBy.lastName,createdBy.pseudonym,createdBy.owner,associatedGroups
0,5629499542036630,2025-05-28T19:30:47Z,Address,RITM0584685 / TASK0881439,2025-05-28T19:30:47Z,False,True,193.163.125.52,HTOC Org,6000,43204519559205009624,NIH,SOAR,APIUser7Lnh3,HTOC Org,None
1,5629499542014600,2025-05-21T13:34:46Z,Address,RITM0582833,2025-05-21T13:34:46Z,False,True,176.65.149.182,HTOC Org,6000,43204519559205009624,NIH,SOAR,APIUser7Lnh3,HTOC Org,None
2,5629499542036631,2025-05-28T19:30:47Z,Address,RITM0584685 / TASK0881439,2025-05-28T19:30:47Z,False,True,193.163.125.127,HTOC Org,6000,43204519559205009624,NIH,SOAR,APIUser7Lnh3,HTOC Org,None
3,6755399482150846,2025-11-26T17:40:45Z,Address,RITM0628389,2025-11-26T17:40:45Z,False,True,64.62.156.173,HTOC Org,6000,43204519559205009624,NIH,SOAR,APIUser7Lnh3,HTOC Org,None
4,6755399481063954,2025-11-17T16:08:11Z,Address,TASK0931507,2025-11-17T16:08:11Z,False,True,65.49.1.52,HTOC Org,6000,43204519559205009624,NIH,SOAR,APIUser7Lnh3,HTOC Org,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1907,5629499577081590,2025-11-17T16:08:10Z,Address,TASK0931507,2025-11-17T16:08:10Z,False,True,204.76.203.30,HTOC Org,6000,43204519559205009624,NIH,SOAR,APIUser7Lnh3,HTOC Org,None
1908,5629499556005846,2025-06-30T12:21:16Z,Address,RITM0594014,2025-06-30T12:21:16Z,False,True,65.49.1.127,HTOC Org,6000,43204519559205009624,NIH,SOAR,APIUser7Lnh3,HTOC Org,None
1909,6755399459033749,2025-06-16T17:42:34Z,Address,RITM0589984,2025-06-16T17:42:34Z,False,True,192.42.116.195,HTOC Org,6000,43204519559205009624,NIH,SOAR,APIUser7Lnh3,HTOC Org,None
1910,5629499541090447,2025-05-14T17:45:29Z,Address,FBI Email Alert May 14 2025,2025-05-14T17:45:29Z,False,True,87.236.176.15,HTOC Org,6000,43204519559205009624,NIH,SOAR,APIUser7Lnh3,HTOC Org,None


In [7]:
import pandas as pd

# Load the CSV from the shared drive, extracting the 'value' column
export_path = r"H:\ThreatConnectExport.csv"
export_df = pd.read_csv(export_path, dtype=str)

# Check for likely column name(s) for the indicator in 'export_df'
value_col = None
for col_candidate in ['value', 'Value', 'indicator', 'summary']:
    if col_candidate in export_df.columns:
        value_col = col_candidate
        break
if value_col is None:
    raise KeyError(f"Could not find indicator column in {export_path}. Columns: {list(export_df.columns)}")

# Set of indicators in the export
export_values = set(export_df[value_col].dropna().astype(str))

# Set of indicators in the NIH SOAR records (using the 'summary' column)
nih_soar_values = set(nih_soar_records['summary'].dropna().astype(str))

# Find indicators in H:\ThreatConnectExport.csv that are not in nih_soar_records
missing_in_nih_soar = sorted(export_values - nih_soar_values)

# Show result
display(f"{len(missing_in_nih_soar)} indicators in {export_path} not present in NIH SOAR export")
display(missing_in_nih_soar)

'126 indicators in H:\\ThreatConnectExport.csv not present in NIH SOAR export'

['103.148.186.191',
 '103.224.212.247',
 '103.70.114.33',
 '104.21.20.66',
 '104.21.45.12',
 '104.21.51.233',
 '104.21.60.48',
 '104.21.66.253',
 '104.21.9.133',
 '104.21.94.157',
 '104.237.131.164',
 '104.237.133.199',
 '108.156.184.111',
 '108.156.184.46',
 '108.156.184.47',
 '108.156.184.9',
 '119.96.59.16',
 '140.245.107.239',
 '142.196.149.38',
 '145.239.200.77',
 '146.70.147.11',
 '154.218.0.108',
 '154.218.0.29',
 '154.218.0.4',
 '154.218.0.95',
 '155.138.160.190',
 '156.226.37.170',
 '157.230.162.215',
 '157.230.221.156',
 '172.234.246.109',
 '172.65.190.172',
 '172.67.137.204',
 '172.67.191.131',
 '172.67.191.164',
 '172.67.210.171',
 '172.93.201.196',
 '18.193.71.144',
 '18.204.117.42',
 '18.204.241.193',
 '18.235.127.223',
 '185.208.158.132',
 '185.27.134.24',
 '185.53.179.128',
 '192.155.84.28',
 '192.64.119.6',
 '193.105.73.21',
 '193.3.19.2',
 '194.36.209.130',
 '194.41.58.26',
 '195.210.46.64',
 '196.251.69.91',
 '2.136.164.95',
 '200.21.254.245',
 '206.119.188.30',
 '21